# Неделя 10 — Fine-tuning BERT

Цель: адаптировать encoder к тональности. Код обучения доступен прямо ниже. Основной воспроизводимый запуск пяти опытов — `python -m nlp_project.transformer`. RUN_TRAINING можно включить для отдельного запуска, он не перезаписывает итоговую таблицу.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'nlp_project').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from nlp_project.core import corpus, clean, tokens, metrics, SEED
df = corpus()
train, val, test = [df[df.split == s].copy() for s in ('train', 'validation', 'test')]


In [ ]:
from nlp_project.transformer import fine_tune, EPOCHS, BATCH_SIZE, MAX_LENGTH
from transformers import AutoTokenizer
print({'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'max_length': MAX_LENGTH, 'optimizer': 'AdamW', 'weight_decay': .01, 'seed': SEED})
RUN_TRAINING = False
if RUN_TRAINING:
    path = ROOT / 'models/pretrained_bert'
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model, history = fine_tune(path, tokenizer, train, val, 3e-5, 1., 'bert_notebook_demo')
    display(pd.DataFrame(history))
history_path = ROOT / 'reports/bert_training_history.csv'
if not history_path.exists():
    raise FileNotFoundError('Fine-tuning ещё не выполнен. Запустите nlp_project.transformer после разрешения huggingface.co.')
display(pd.read_csv(history_path))
display(pd.read_csv(ROOT / 'reports/all_metrics.csv').query("model in ['bert_frozen_lr', 'bert_lr3e5_full']"))


## Выводы

Выбор checkpoint только по validation macro-F1. До/после сравниваются frozen encoder + LR и fine-tuned BERT. Рост качества не гарантирован; отрицательный результат тоже нужно честно отразить.